In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [10]:
data = pd.read_csv("/content/glass.csv")

# Features and target
X = data.drop("Type", axis=1).values
X = data.drop("Type", axis=1).values
# Target
y = data["Type"].values

# Convert labels to continuous values
unique_classes = np.unique(y)
class_map = {label: idx for idx, label in enumerate(unique_classes)}
y = np.array([class_map[i] for i in y])


In [12]:
# Train Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [13]:
# Feature Scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# One Hot Encoding
# -----------------------------
num_classes = len(np.unique(y))

def one_hot(y, classes):
    encoded = np.zeros((len(y), classes))
    encoded[np.arange(len(y)), y] = 1
    return encoded

y_train_encoded = one_hot(y_train, num_classes)


In [14]:
# Softmax Function
# -----------------------------
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# -----------------------------
# Model Parameters
# -----------------------------
n_features = X_train.shape[1]
weights = np.zeros((n_features, num_classes))
bias = np.zeros((1, num_classes))

learning_rate = 0.01
epochs = 1000


In [15]:
# Training (Gradient Descent)
# -----------------------------
for epoch in range(epochs):

    # Linear output
    z = np.dot(X_train, weights) + bias

    # Softmax prediction
    y_pred = softmax(z)

    # Loss gradient
    error = y_pred - y_train_encoded

    dw = np.dot(X_train.T, error) / X_train.shape[0]
    db = np.sum(error, axis=0, keepdims=True) / X_train.shape[0]

    # Update weights
    weights -= learning_rate * dw
    bias -= learning_rate * db

    if epoch % 100 == 0:
        loss = -np.mean(y_train_encoded * np.log(y_pred + 1e-9))
        print(f"Epoch {epoch}, Loss: {loss:.4f}")


Epoch 0, Loss: 0.2986
Epoch 100, Loss: 0.2473
Epoch 200, Loss: 0.2195
Epoch 300, Loss: 0.2029
Epoch 400, Loss: 0.1920
Epoch 500, Loss: 0.1844
Epoch 600, Loss: 0.1788
Epoch 700, Loss: 0.1743
Epoch 800, Loss: 0.1708
Epoch 900, Loss: 0.1679


In [16]:
# Prediction
# -----------------------------
def predict(X):
    z = np.dot(X, weights) + bias
    probs = softmax(z)
    return np.argmax(probs, axis=1)

y_pred = predict(X_test)

# -----------------------------
# Accuracy
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", accuracy)


Test Accuracy: 0.5813953488372093
